---
# Weather Data Extraction – ETL Pipeline

This notebook is responsible for the **data extraction stage** of the ETL pipeline.

The goal is to retrieve historical daily weather data for Winnipeg (2023) using the Open-Meteo API.  
This data will later be cleaned, transformed, and integrated with public transit performance data for analysis.

---

## Data Source

Weather data is retrieved from the **Open-Meteo Historical Weather API**.

- Provider: Open-Meteo
- Data Type: Historical daily weather observations
- Location: Winnipeg, Canada
- Time Range: January 1, 2023 – December 31, 2023

## Variables Collected

The following daily weather variables are extracted:

- Weather code (general weather condition)
- Maximum temperature (°C)
- Minimum temperature (°C)
- Mean temperature (°C)
- Total precipitation (mm)
- Mean visibility
- Mean wind speed (km/h)

These variables were selected based on their potential impact on public transportation performance (e.g., delays caused by extreme weather).


In [1]:
import openmeteo_requests
import requests_cache
from retry_requests import retry
import pandas as pd

In [3]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 49.8844,
	"longitude": -97.147,
	"start_date": "2023-01-01",
	"end_date": "2023-12-31",
	"daily": ["weather_code", "temperature_2m_max", "precipitation_sum", "temperature_2m_mean", "temperature_2m_min", "visibility_mean", "wind_speed_10m_mean"],
	"timezone": "America/Chicago",
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_weather_code = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_max = daily.Variables(1).ValuesAsNumpy()
daily_precipitation_sum = daily.Variables(2).ValuesAsNumpy()
daily_temperature_2m_mean = daily.Variables(3).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(4).ValuesAsNumpy()
daily_visibility_mean = daily.Variables(5).ValuesAsNumpy()
daily_wind_speed_10m_mean = daily.Variables(6).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

daily_data["weather_code"] = daily_weather_code
daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["precipitation_sum"] = daily_precipitation_sum
daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["visibility_mean"] = daily_visibility_mean
daily_data["wind_speed_10m_mean"] = daily_wind_speed_10m_mean

df = pd.DataFrame(data = daily_data)
print("\nDaily data\n", df)


Coordinates: 49.876976013183594°N -97.20001220703125°E
Elevation: 236.0 m asl
Timezone: b'America/Chicago'b'GMT-6'
Timezone difference to GMT+0: -21600s

Daily data
                          date  weather_code  temperature_2m_max  \
0   2023-01-01 00:00:00+00:00          71.0               -6.30   
1   2023-01-02 00:00:00+00:00           3.0               -7.00   
2   2023-01-03 00:00:00+00:00           3.0               -5.00   
3   2023-01-04 00:00:00+00:00           3.0               -7.65   
4   2023-01-05 00:00:00+00:00           3.0               -7.15   
..                        ...           ...                 ...   
360 2023-12-27 00:00:00+00:00           1.0                1.20   
361 2023-12-28 00:00:00+00:00           1.0                0.05   
362 2023-12-29 00:00:00+00:00           3.0                0.70   
363 2023-12-30 00:00:00+00:00          73.0               -3.90   
364 2023-12-31 00:00:00+00:00           3.0               -6.15   

     precipitation_sum  tempe

## Output

The extracted raw data is saved as a CSV file in the **data_raw** folder.  
No cleaning or transformations are performed in this step to preserve the original source data.

In [4]:
df.to_csv("weather_raw.csv")